In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium

In [ ]:
df_election = pd.read_excel('src_election/election_1ertour_2022/resultats-par-niveau-burvot-t1-france-entiere.xlsx')

In [ ]:
df_election.isna().any()

In [ ]:
df_election["Votants"].sum().item()

In [ ]:
print(df_election.columns)

### Certaines colonnes, importées sous l'intitulé générique Unnamed, renferment des informations pertinentes. Leur renommage est nécessaire pour les identifier et les manipuler correctement. 

In [ ]:
# Compte le nombre de lignes dupliquées
nb_doublons = df_election.duplicated().sum()
print(f"Nombre de doublons : {nb_doublons}")

###  Nous constatons qu'il n'y a aucune valeur nulle ou NaN ; cependant, un détail sera abordé ultérieurement.

In [ ]:
#Solution la plus propre a garder !!!

# 1. On isole les informations géographiques et globales (les 18 premières colonnes environ)
# On garde juste ce qui nous intéresse pour l'analyse
cols_fixes = ['Code du département', 'Libellé du département', 'Code de la commune', 'Libellé de la commune', 'Inscrits', 'Abstentions', 'Exprimés', 'Blancs', 'Nuls', 'Votants']
df_clean = df_election[cols_fixes].copy()

# 2. On repère où commencent les candidats
# Dans ton fichier, le premier candidat commence à la colonne "N°Panneau" (index 21 si on compte depuis 0)
# Chaque candidat prend 7 colonnes. 
index_debut_candidats = df_election.columns.get_loc('N°Panneau')

# 3. La boucle magique de nettoyage
# Il y a 12 candidats, on va faire des "bonds" de 7 colonnes
for i in range(12):
    # L'index de la colonne "Nom" pour ce candidat
    idx_nom = index_debut_candidats + 2 + (i * 7)
    # L'index de la colonne "Voix" pour ce candidat
    idx_voix = index_debut_candidats + 4 + (i * 7)
    
    # On récupère le nom du candidat (en prenant la première ligne, car c'est le même partout)
    nom_candidat = df_election.iloc[0, idx_nom]
    
    # On crée dynamiquement la colonne dans notre nouveau DataFrame propre
    # ex: df_clean['voix_MACRON'] = df_election.iloc[:, idx_voix]
    df_clean[f'voix_{nom_candidat}'] = df_election.iloc[:, idx_voix].astype(int)

# On affiche le résultat
df_clean.head()

In [ ]:
print('Résultat Macron:', df_clean['voix_MACRON'].sum().item(), 'voix exprimées')
print('Résultat Le Pen:', df_clean['voix_LE PEN'].sum().item(), 'voix exprimées')
print('Résultat Melenchon:', df_clean['voix_MÉLENCHON'].sum().item(), 'voix exprimées')

In [ ]:

df_election[(df_election['Exprimés'] == 0) | (df_election['Votants'] == 0) | (df_election['Inscrits'] == 0)]

In [ ]:
# Voir la répartition par département de ces lignes à 0
df_election[df_election['Exprimés'] == 0]['Libellé du département'].value_counts().plot(kind='bar')

### Les bureaux de vote affichant 0 voix exprimées sont retirés du dataset : l'absence de données électorales exploitables sur ces lignes introduirait du bruit et fausserait l'apprentissage du modèle.

In [ ]:
# ==========================================
# ÉTAPE 1 : AGRÉGATION (Fusion des bureaux de vote)
# ==========================================

# On repère automatiquement toutes les colonnes de résultats (celles qui commencent par 'voix_')
cols_voix = [col for col in df_clean.columns if col.startswith('voix_')]

# On définit toutes les colonnes qu'on veut additionner
cols_numeriques = ['Inscrits', 'Abstentions', 'Votants', 'Blancs', 'Nuls', 'Exprimés']
cols_a_sommer = cols_numeriques + cols_voix

# Le groupby : On rassemble toutes les lignes qui ont la même commune.
# Le .reset_index() à la fin est crucial : il transforme le résultat en un DataFrame classique
df_communes = df_clean.groupby([
    'Code du département', 
    'Libellé du département', 
    'Code de la commune', 
    'Libellé de la commune'
])[cols_a_sommer].sum().reset_index()


# ==========================================
# ÉTAPE 2 : CRÉATION DU CODE INSEE (La clé de voûte)
# ==========================================

# On s'assure que le département est au format texte (string)
dept = df_communes['Code du département'].astype(str)

# On s'assure que le code commune est au format texte ET fait exactement 3 caractères (.str.zfill(3))
# Exemple : "4" devient "004", "12" devient "012", "154" reste "154"
comm = df_communes['Code de la commune'].astype(str).str.zfill(3)

# On concatène (fusionne) les deux pour créer la clé officielle
df_communes['Code_INSEE'] = dept + comm

# Juste avant de calculer les pourcentages à l'étape 3 :
initial_count = len(df_communes)

# On ne garde que les communes qui ont au moins 1 inscrit ET au moins 1 exprimé
df_communes = df_communes[(df_communes['Inscrits'] > 0) & (df_communes['Exprimés'] > 0)].copy()

deleted_count = initial_count - len(df_communes)
print(f"Nettoyage : {deleted_count} communes supprimées (données incomplètes ou nulles).")

# ==========================================
# ÉTAPE 3 : CALCUL DES POURCENTAGES (Pour le Machine Learning)
# ==========================================

# On boucle sur la liste des candidats qu'on a créée à l'étape 1
for col_voix in cols_voix:
    # On génère un nouveau nom de colonne. Ex: 'voix_MACRON' devient 'pct_MACRON'
    nom_col_pct = col_voix.replace('voix_', 'pct_')
    
    # La formule mathématique vectorisée
    # On gère le cas (très rare) où Exprimés = 0 pour éviter une division par zéro qui ferait planter le code
    df_communes[nom_col_pct] = (df_communes[col_voix] / df_communes['Exprimés'] * 100).fillna(0).round(2)


# ==========================================
# ÉTAPE 4 : NETTOYAGE FINAL (L'esthétique)
# ==========================================

# On réorganise les colonnes pour mettre le Code_INSEE tout devant, c'est plus pratique à lire
toutes_les_colonnes = df_communes.columns.tolist()
# On enlève le Code_INSEE de sa position actuelle (tout à la fin)...
toutes_les_colonnes.remove('Code_INSEE')
# ... pour le remettre au tout début
colonnes_finales = ['Code_INSEE'] + toutes_les_colonnes

df_final = df_communes[colonnes_finales]


df_final.head().columns

### Nous allons construire le DataFrame prêt à la fusion avec les données INSEE, en agrégeant la cible par familles politiques (Gauche, Centre, Droite) et en conservant les clés de jointure nécessaires.

In [ ]:
# On prépare la base avec les chiffres bruts nécessaires
df_blocs = df_final[[
    'Code_INSEE', 
    'Libellé de la commune', 
    'Inscrits', 
    'Abstentions', 
    'Votants', 
    'Blancs', 
    'Nuls', 
    'Exprimés'
]].copy()

# 1. L'abstention (sur les inscrits)
df_blocs['pct_abstention'] = (df_blocs['Abstentions'] / df_blocs['Inscrits'] * 100).round(2)

# 2. Le vote blanc (sur les votants) - Geste politique de refus
df_blocs['pct_blancs'] = (df_blocs['Blancs'] / df_blocs['Votants'] * 100).round(2)

# 3. Le vote nul (sur les votants) - Souvent une erreur ou une radiation
df_blocs['pct_nuls'] = (df_blocs['Nuls'] / df_blocs['Votants'] * 100).round(2)

groupes = {
    'gauche': ['voix_ARTHAUD', 'voix_ROUSSEL', 'voix_MÉLENCHON', 'voix_HIDALGO', 'voix_JADOT', 'voix_POUTOU'],
    'centre': ['voix_MACRON', 'voix_LASSALLE'],
    'droite': ['voix_LE PEN', 'voix_ZEMMOUR', 'voix_PÉCRESSE', 'voix_DUPONT-AIGNAN']
}

for nom_bloc, liste_candidats in groupes.items():
    # On fait la somme des voix du bloc pour chaque commune
    total_voix_bloc = df_final[liste_candidats].sum(axis=1)
    # On calcule le pourcentage sur les EXPRIMÉS
    df_blocs[f'pct_{nom_bloc}'] = (total_voix_bloc / df_blocs['Exprimés'] * 100).round(2)

In [ ]:
df_blocs.head()

In [ ]:
# Export des datasets
df_blocs.to_csv('src/election.csv', index=False, sep=';', encoding='utf-8')

In [ ]:
# On prépare les données : on fait la moyenne des % de chaque commune
stats_blocs = df_blocs[['pct_gauche', 'pct_centre', 'pct_droite']].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=stats_blocs.index, y=stats_blocs.values, palette=['coral', 'royalblue', 'gold'])
plt.title('Moyenne des scores par bloc (par commune)')
plt.ylabel('Pourcentage moyen (%)')
plt.show()

In [ ]:
# Exemple avec le bloc Droite 
sns.jointplot(data=df_blocs, x='pct_abstention', y='pct_droite', kind="hex", color="#4CB391")
plt.subplots_adjust(top=0.9)
plt.suptitle('Relation entre Abstention et Vote à Droite')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_blocs[['pct_gauche', 'pct_centre', 'pct_droite']])
plt.title('Dispersion des votes par bloc sur l\'ensemble des communes')
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
sns.scatterplot(data=df_blocs, x='pct_gauche', y='pct_droite', alpha=0.3, size='Exprimés', sizes=(10, 200))
plt.title('Opposition Gauche vs Droite (la taille des points = taille de la commune)')
plt.show()

In [ ]:
# On sélectionne les colonnes numériques
cols_etude = ['pct_abstention', 'pct_blancs', 'pct_gauche', 'pct_centre', 'pct_droite']
corr = df_blocs[cols_etude].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Matrice de corrélation des comportements de vote')
plt.show()

In [ ]:
import pandas as pd
import requests
import folium

# --- 1. Changement de référentiel géographique ---
url_alt = "https://geo.api.gouv.fr/communes?fields=nom,code,centre"
response = requests.get(url_alt)
data = response.json()

coords_list = []
for c in data:
    if 'centre' in c:
        coords_list.append({
            'code_insee_ref': c['code'],
            'nom_officiel': c['nom'],
            'latitude': c['centre']['coordinates'][1],
            'longitude': c['centre']['coordinates'][0]
        })

df_coords_alt = pd.DataFrame(coords_list)

# --- 2. PRÉPARATION DES DONNÉES ÉLECTORALES ---
df_blocs['Code_INSEE'] = df_blocs['Code_INSEE'].astype(str).str.strip().str.zfill(5)

# --- 3. FUSION (MERGE) ---
df_final = pd.merge(
    df_blocs, 
    df_coords_alt, 
    left_on='Code_INSEE', 
    right_on='code_insee_ref', 
    how='left',
    indicator=True
)

# --- 4. DIAGNOSTIC DES MANQUANTS ---
manquants = df_final[df_final['_merge'] == 'left_only']
if not manquants.empty:
    print(f" Communes non localisées : {len(manquants)}")
    print(manquants[['Code_INSEE']].head(5))

# On ne garde que ce qui est localisé pour la carte
df_map = df_final[df_final['_merge'] == 'both'].copy()

# --- 5. CALCULS DES COULEURS ET OPACITÉ ---
def get_color(row):
    scores = {'red': row['pct_gauche'], 'orange': row['pct_centre'], 'blue': row['pct_droite']}
    return max(scores, key=scores.get)

df_map['couleur_gagnante'] = df_map.apply(get_color, axis=1)
df_map['score_max'] = df_map[['pct_gauche', 'pct_centre', 'pct_droite']].max(axis=1)

# --- 6. CRÉATION DE LA CARTE ---
m = folium.Map(location=[46.2276, 2.2137], zoom_start=6, tiles='cartodbpositron')

# Utilisation de itertuples pour plus de rapidité
for row in df_map.itertuples():
    folium.CircleMarker(
        location=[row.latitude, row.longitude],
        radius=1.8,
        color=None,
        fill=True,
        fill_color=row.couleur_gagnante,
        fill_opacity=min(row.score_max / 100 * 1.2, 1.0), 
        tooltip=(f"<b>{row.nom_officiel}</b> ({row.Code_INSEE})<br>"
                 f"Gauche: {row.pct_gauche}%<br>"
                 f"Centre: {row.pct_centre}%<br>"
                 f"Droite: {row.pct_droite}%")
    ).add_to(m)

m.save("carte_electorale_finale.html")
print(f"Carte générée avec {len(df_map)} communes.")